# Base Model vs Chat Model (TinyLlama 1.1B)

We give the **same question** to two versions of the same model:

- **TinyLlama-1.1B (base)** : pre-trained on 3 trillion tokens of web text and code. It only learned to predict the next word.
- **TinyLlama-1.1B-Chat** : the same base model, trained further in two steps:
  1. **SFT** on UltraChat (example conversations generated by ChatGPT)
  2. **DPO** on UltraFeedback (answers ranked better/worse by GPT-4)

```
Random weights → pre-training (3T tokens) → BASE → SFT (UltraChat) → DPO (UltraFeedback) → CHAT
```

Watch two things in the outputs:
1. Does the model **answer** the question, or **continue the text**?
2. Does the model **stop** when it is done, or keep going until the token limit?

Runtime: use a GPU (Runtime → Change runtime type → T4 GPU).

## Step 1: Setup

In [1]:
import torch  # PyTorch: the library that does the model's calculations
from transformers import AutoTokenizer, AutoModelForCausalLM  # Hugging Face tools to download a tokenizer and a model
# AutoModelForCausalLM is the Hugging Face class that downloads a model and sets it up for text generation.


device = "cuda" if torch.cuda.is_available() else "cpu"  # use the GPU if Colab gave us one, otherwise use the CPU
print("Running on:", device)  # show which one we are using

Running on: cuda


AutoModelForCausalLM is the Hugging Face class that downloads a model and sets it up for text generation. Its name has three parts.

### "Auto"

It picks the correct model class by itself. When you call from_pretrained(base_name):

downloads config.json            → says "this is a Llama model"
   → Auto picks LlamaForCausalLM
   → downloads the weights file (model.safetensors)
   → builds the model and fills in the weights

Give it a Qwen name and it picks Qwen2ForCausalLM instead. The same line of code works for any model family.

### "Causal LM" (Causal Language Model)

A model that predicts the next token, reading left to right. Each token can only look at the tokens before it, never at the tokens after it (the causal mask from the attention-mask topic). All text-generation models (GPT, Llama, Qwen, TinyLlama) are causal language models.

### "For"

It loads the model with its output layer: the last layer that turns the model's numbers into a score for every token in the vocabulary (32,000 scores for TinyLlama). The highest score is the predicted next token. Without this layer, the model cannot generate text.




## Step 2: The question we will ask both models

In [2]:
prompt = "What is NAV in mutual funds?"  # the same question goes to both models

## Step 3: Load the BASE model

In [ ]:
base_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# the base model: only pre-trained (predict the next word
#intermediate-step-1431k: a copy of the model saved during pre-training, at training step 1,431,000


base_tokenizer = AutoTokenizer.from_pretrained(base_name)
# Downloads the tokenizer files from this model's Hugging Face page and builds the tokenizer. This is what turns text into token IDs and back.

base_model = AutoModelForCausalLM.from_pretrained(base_name).to(device)
'''
from_pretrained(base_name): downloads config.json (the model's design: number of layers, vocabulary size, etc.)
and the weights file, builds a LlamaForCausalLM, and fills in the 1.1 billion weights.
.to(device): moves all the weights to the GPU (or CPU, if there's no GPU).
The model has to be on the same device as the input token IDs.
'''

print("Base model loaded")  # confirm loading finished

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded


## Step 4: Look at the tokenizer

In [ ]:

print(type(base_tokenizer))  # which tokenizer class AutoTokenizer picked (a Llama tokenizer)
print("Vocabulary size:", len(base_tokenizer))  # how many different tokens it knows (about 32,000)
print("Tokens:", base_tokenizer.tokenize(prompt))  # the question split into tokens; "▁" means there was a space before the token
print("Token IDs:", base_tokenizer.encode(prompt))  # the numbers the model receives; ID 1 at the start is <s>, the start-of-text token

<class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>
Vocabulary size: 32000
Tokens: ['▁What', '▁is', '▁N', 'AV', '▁in', '▁mut', 'ual', '▁funds', '?']
Token IDs: [1, 1724, 338, 405, 7520, 297, 5478, 950, 29199, 29973]


**What to notice:** this tokenizer shows a space as `▁`. Qwen's tokenizer showed it as `Ġ`. Different model families use different tokenizers, which is why the tokenizer must always be loaded from the same model name as the model.

## Step 5: Ask the BASE model

In [ ]:
# Convert the question into token IDs (numbers the model understands), as a PyTorch tensor.
# .to(device) moves those IDs to the GPU, where the model is.
inputs = base_tokenizer(prompt, return_tensors="pt").to(device)

# Ask the model to write new text after the question, one token at a time.
# Each new token is added to the input before the next one is predicted.
output_ids = base_model.generate(

    # Pass in the question's token IDs and the attention mask.
    # ** unpacks the dictionary, same as writing input_ids=..., attention_mask=...
    **inputs,

    # Upper limit: stop after 80 new tokens, even if the model has not finished.
    # The model can stop earlier by itself if it generates </s>.
    max_new_tokens=80,

    # Always pick the highest-scoring next token instead of picking randomly.
    # This makes the output the same every time the cell is run.
    do_sample=False,
)

# The output contains the question's 10 tokens followed by the new tokens.
# [10:] skips the question and keeps only the newly generated tokens.
new_ids = output_ids[0][inputs["input_ids"].shape[1]:]

# Print the original question.
# Shown first, so the output below can be compared with it.
print("QUESTION:", prompt)

# Print a heading for the model's output.
# \n adds a blank line before the heading.
print("\nBASE MODEL OUTPUT:")

# Convert the new token IDs back into readable text.
# skip_special_tokens=True hides markers like <s> and </s>.
print(base_tokenizer.decode(new_ids, skip_special_tokens=True))

# Print how many new tokens were generated.
# 80 means the model never stopped by itself; less than 80 means it generated </s>.
print("\nNew tokens generated:", len(new_ids), "out of a limit of 80")



'''
Output :
1. It repeats the question, because it is a base model.
It was only trained to continue text. It treats "What is NAV in mutual funds?" as the start of a document, not as a question to answer.

2. It writes "Question? Answer." repeatedly, because that pattern is common in its training data.
FAQ and SEO web pages often repeat a question with its answer.
The model is continuing the text in the format it saw most often after a line like this.

3. It repeats the same line, because of do_sample=False.
It always picks the single highest-scoring token.
Once the same line has appeared twice, writing it a third time becomes the highest-scoring continuation,
so it gets stuck in a loop.

4. It stops mid-sentence at "fund's", because it never generated </s>.
It was never trained to end an answer with EOS (that is taught in SFT).
So the only thing that stopped it was the 80-token limit.

What is correct: "NAV is the net asset value... assets minus the liabilities" is right.
The knowledge came from pre-training. What's missing is the behaviour: answering once and stopping, which comes from SFT and DPO.

'''

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What is NAV in mutual funds?

BASE MODEL OUTPUT:

What is NAV in mutual funds?
What is NAV in mutual funds? NAV is the net asset value of a mutual fund. It is the value of the fund's assets minus the liabilities.
What is NAV in mutual funds? NAV is the net asset value of a mutual fund. It is the value of the fund's

New tokens generated: 80 out of a limit of 80


**What to notice:** the base model treats our question as the start of a document and writes what would come next, and usually runs until the 80-token limit.

## Step 6: Load the CHAT model

In [ ]:
chat_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # the same base model after SFT + DPO
chat_tokenizer = AutoTokenizer.from_pretrained(chat_name)  # download this model's tokenizer
chat_model = AutoModelForCausalLM.from_pretrained(chat_name).to(device)  # download the chat model's weights and move them to the GPU/CPU
print("Chat model loaded")  # confirm loading finished

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Chat model loaded


## Step 7: Put the question in the chat format

In [ ]:
messages = [{"role": "user", "content": prompt}]  # chat models expect a conversation: a list of messages, each with a role and content

chat_text = chat_tokenizer.apply_chat_template(  # wrap our question in the exact format the model saw during SFT
    messages,                     # the conversation to format
    tokenize=False,               # return plain text so we can print and read it
    add_generation_prompt=True,   # add the "assistant" marker at the end, so the model knows it is its turn to answer
)

print(chat_text)  # show the exact text the chat model will receive

'''

This is the same format as every conversation in TinyLlama Chat's SFT training data.
The model learned from about 200,000 conversations that:

after <|user|> comes a question,
after <|assistant|> comes an answer to that question,
and the answer ends with </s>.

Giving it the same format makes it do what it was trained to do: write an answer, then </s>.

'''

<|user|>
What is NAV in mutual funds?</s>
<|assistant|>



**What to notice:** our question is now wrapped in markers: `<|user|>` before the question, and `<|assistant|>` at the end. During SFT, the model learned that after `<|assistant|>` it should write an answer, and then write `</s>` (its end token) to stop.

This template is different from Qwen's (`<|im_start|>user`). Every model family has its own.

## Step 8: Ask the CHAT model

In [ ]:
inputs = chat_tokenizer(chat_text, return_tensors="pt", add_special_tokens=False).to(device)  # turn the chat text into token IDs; the template already has what it needs, so no extra tokens are added

output_ids = chat_model.generate(  # ask the model to produce new tokens
    **inputs,             # pass in the token IDs
    max_new_tokens=80,    # same limit as the base model, so the comparison is fair
    do_sample=False,      # same setting as the base model: always pick the most likely token
)

new_ids = output_ids[0][inputs["input_ids"].shape[1]:]  # keep only the newly generated tokens

print("QUESTION:", prompt)  # show the question
print("\nCHAT MODEL OUTPUT:")  # heading for the output
print(chat_tokenizer.decode(new_ids, skip_special_tokens=True))  # convert the new token IDs back into text
print("\nNew tokens generated:", len(new_ids), "out of a limit of 80")  # if below 80, the model decided by itself that the answer was finished



'''
What happens in Chat model over base model ?

The steps are the same as for the base model. What changes is the input and how the model behaves.

Step by step

1. Question → chat template

"What is NAV in mutual funds?"
   → <|user|>
     What is NAV in mutual funds?</s>
     <|assistant|>

The base model got the plain question. The chat model gets the question in the conversation format it was trained on.

2. Chat template → token IDs

The tokenizer converts the whole formatted text, including <|user|>, </s> and <|assistant|>, into token IDs. So the input is longer than the base model's 10 tokens.

3. Generate, one token at a time

The model reads: ... <|assistant|>
   → SFT taught it: after <|assistant|> comes an answer
   → predicts the first word of an answer (not a repeat of the question)
   → keeps predicting the answer, one token at a time
   → answer is complete → SFT taught it: now write </s>
   → predicts </s> → generate() stops




'''

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What is NAV in mutual funds?

CHAT MODEL OUTPUT:
NAV (Net Asset Value) is a measure of the value of a mutual fund's assets, including its investments, liabilities, and other financial obligations. It is calculated by dividing the fund's total assets by the number of shares outstanding. NAV is used to determine the price of a fund's shares, which is the price at which investors

New tokens generated: 80 out of a limit of 80


## Step 9: Try more prompts on both models

In [ ]:
more_prompts = [  # a few different kinds of input to compare
    "Write a one-line definition of SIP.",                              # an instruction
    "The three things to check before investing in a mutual fund are",  # an unfinished sentence
    "List two benefits of investing through a SIP.",                     # a task
]

for p in more_prompts:  # repeat the comparison for each prompt
    base_inputs = base_tokenizer(p, return_tensors="pt").to(device)  # base model gets the plain text
    base_out = base_model.generate(**base_inputs, max_new_tokens=60, do_sample=False)  # generate up to 60 new tokens
    base_new = base_out[0][base_inputs["input_ids"].shape[1]:]  # keep only the new tokens

    chat = chat_tokenizer.apply_chat_template(  # chat model gets the chat format
        [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    chat_inputs = chat_tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(device)  # turn it into token IDs
    chat_out = chat_model.generate(**chat_inputs, max_new_tokens=60, do_sample=False)  # generate up to 60 new tokens
    chat_new = chat_out[0][chat_inputs["input_ids"].shape[1]:]  # keep only the new tokens

    print("=" * 70)  # separator line between prompts
    print("PROMPT:", p)  # show the prompt
    print("\n--- BASE (", len(base_new), "tokens ) ---")  # base output heading with its token count
    print(base_tokenizer.decode(base_new, skip_special_tokens=True))  # base model's text
    print("\n--- CHAT (", len(chat_new), "tokens ) ---")  # chat output heading with its token count
    print(chat_tokenizer.decode(chat_new, skip_special_tokens=True))  # chat model's text

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Write a one-line definition of SIP.

--- BASE ( 60 tokens ) ---


### 1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1

--- CHAT ( 20 tokens ) ---
SIP: Secure Internet Protocol is a protocol used for secure communication over IP networks.


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: The three things to check before investing in a mutual fund are

--- BASE ( 60 tokens ) ---
: 1.
The three things to check before investing in a mutual fund are: 1.
The three things to check before investing in a mutual fund are: 1.
The three things to check before investing in a mutual fund are: 1

--- CHAT ( 60 tokens ) ---
1. Fund Manager: The first thing to check before investing in a mutual fund is the fund manager. The fund manager is responsible for managing the fund's assets and investing them in the best possible way. The fund manager should have a track record of success and be well-


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: List two benefits of investing through a SIP.

--- BASE ( 60 tokens ) ---

1. You can invest in a mutual fund that is designed to provide you with a regular income.
2. You can invest in a mutual fund that is designed to provide you with a regular capital appreciation.
3. You can invest in a mutual fund that is designed

--- CHAT ( 60 tokens ) ---
1. Tax-saving: SIPs offer tax benefits as they are invested in mutual funds, which are taxed at the long-term capital gains rate. This means that the investor can save tax on the principal amount invested in the SIP.

2


## Discussion questions

1. Which model answered the question, and which one continued the text?
2. Which model used all the allowed tokens, and which one stopped by itself? Why?
3. Both models have the same size, architecture and tokenizer. What made them behave differently?
4. The chat model went through SFT and then DPO. What did each step add?
5. If you want to fine-tune a model for your own task with 1,000 examples, which one would you start from, and why?